# Row-level security, step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will create roles for plant operators, define RLS policies, and verify that different operators see different data from the same query.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup, connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

SCHEMA = os.environ.get('LB_SCHEMA', 'factory')
print('connected; schema:', SCHEMA)

## Step 1, Create roles for plant operators

Create restricted roles for each operator at a real plant (Detroit, Austin, Munich, the plants in the bank seed). Grant schema access and membership so we can SET ROLE into them to demonstrate RLS enforcement. The SQL:

```sql
SELECT 1 FROM pg_roles WHERE rolname = 'op_detroit';
CREATE ROLE op_detroit;
GRANT USAGE ON SCHEMA factory TO op_detroit;
GRANT SELECT ON bank.account TO op_detroit;
GRANT op_detroit TO current_user;
```

In [ ]:
# Operator roles, one per real plant in the bank seed.
operators = {"op_detroit": "BR-NYC", "op_austin": "BR-SFO", "op_munich": "BR-LON"}

me_rows = run_sql("SELECT current_user")
me = me_rows[0][0]

with connect() as conn:
    with conn.cursor() as cur:
        for role, plant in operators.items():
            # Check if role exists via catalog, since CREATE ROLE IF NOT EXISTS is invalid in Postgres
            cur.execute("SELECT 1 FROM pg_roles WHERE rolname = %s", (role,))
            if cur.fetchone():
                print(f"  role already exists: {role}")
            else:
                cur.execute(f"CREATE ROLE {role}")
                print(f"  created role: {role}")
            cur.execute(f"GRANT USAGE ON SCHEMA factory TO {role}")
            cur.execute(f"GRANT SELECT ON bank.account TO {role}")
            # membership so we can SET ROLE into it to demonstrate enforcement
            cur.execute(f'GRANT {role} TO "{me}"')
        print("  granted schema/table SELECT and membership to all operator roles")

## Step 2, Create and enable RLS policy

Enable RLS on the account table with both ENABLE and FORCE (FORCE makes the policy apply to the table owner too). Then create a policy that filters rows based on the plant context variable. The SQL:

```sql
ALTER TABLE bank.account ENABLE ROW LEVEL SECURITY;
ALTER TABLE bank.account FORCE ROW LEVEL SECURITY;
DROP POLICY IF EXISTS machine_operator_isolation ON bank.account;
CREATE POLICY machine_operator_isolation ON bank.account
  FOR SELECT
  USING (branch_id = current_setting('app.current_operator_plant', true))
```

In [ ]:
# ENABLE turns policies on, but the table OWNER bypasses them unless we also
# FORCE row-level security. Without FORCE, the owner sees all rows and the demo
# looks like it does nothing. FORCE makes the policy apply to the owner too.
sql_enable = ("ALTER TABLE bank.account ENABLE ROW LEVEL SECURITY;"
              "ALTER TABLE bank.account FORCE ROW LEVEL SECURITY;")
# Postgres has no CREATE POLICY IF NOT EXISTS; drop any prior one, then create.
sql_drop = "DROP POLICY IF EXISTS machine_operator_isolation ON bank.account;"
sql_policy = """
CREATE POLICY machine_operator_isolation ON bank.account
  FOR SELECT
  USING (branch_id = current_setting('app.current_operator_plant', true))
"""

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql_enable)
        cur.execute(sql_drop)
        cur.execute(sql_policy)
        print("  RLS enabled on bank.account")
        print("  Policy created: machine_operator_isolation")

## Step 3, Same query, different operators

Set each operator role, set its plant context, and run the same SELECT. The database returns only that operator's machines, enforced by the policy. We SET ROLE into each operator because RLS is only enforced for a role that does NOT have BYPASSRLS (and is not the table owner). The query is identical; only the context changes. The SQL:

```sql
SET ROLE op_detroit;
SET app.current_operator_plant = 'BR-NYC';
SELECT account_id, branch_id, status FROM bank.account ORDER BY account_id;
RESET ROLE;
RESET app.current_operator_plant;
```

In [ ]:
# Operator roles, one per real plant in the bank seed.
operators = {"op_detroit": "BR-NYC", "op_austin": "BR-SFO", "op_munich": "BR-LON"}

sql = "SELECT account_id, branch_id, status FROM bank.account ORDER BY account_id"

with connect() as conn:
    with conn.cursor() as cur:
        for role, plant in operators.items():
            cur.execute(f"SET ROLE {role}")
            cur.execute(f"SET app.current_operator_plant = '{plant}'")
            cur.execute(sql)
            rows = cur.fetchall()
            cur.execute("RESET ROLE")
            cur.execute("RESET app.current_operator_plant")
            print(f"\n  as {role} (plant {plant}): {len(rows)} machines, the SAME query:")
            for row in rows:
                print(f"    {row}")

## Step 4, EXPLAIN shows RLS policy in query plan

Run EXPLAIN to see the RLS filter in the query plan. The SQL:

```sql
SET app.current_operator_plant = 'PLANT-ID';
EXPLAIN SELECT account_id FROM bank.account WHERE health_pct > 50
```

In [ ]:
plants = run_sql("SELECT branch_id FROM bank.branch LIMIT 1")
if not plants:
    print("No plants found")
else:
    branch_id = plants[0][0]

    sql = "EXPLAIN SELECT account_id FROM bank.account WHERE health_pct > 50"

    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SET app.current_operator_plant = '{branch_id}'")
            cur.execute(sql)
            rows = cur.fetchall()
            print("Query plan (should include RLS filter):")
            for row in rows:
                print(f"  {row[0]}")

## What you learned

You implemented row-level security (RLS) for multi-tenant access control. Roles, policies, and context variables work together so operators see only their plant's data. RLS is enforced at the database level, not the application.

Next: **No double-booking** uses exclusion constraints to prevent overlapping settlements on the same account.